<a href="https://colab.research.google.com/github/matzz-11/Quantum_Colab.ipynb/blob/main/5_Barreira_Fun%C3%A7%C3%A3o_Delta_de_Dirac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Teoria**

---

## **Potencial, Função de onda e Energia**

Vamos trabalhar aqui com um caso especial do potencial Delta de Dirac, apenas trocando o sinal de $-\alpha$ para $\alpha$, de forma que:

$$
V(x) = αδ(x), α > 0
$$

Sem esse sinal não temos mais o estado ligado, porém **os coeficientes de transmissão e reflexão permanecem inalterados**. Ou seja, vemos que de alguma forma a partícula possui a **mesma probabilidade** de atravessar o poço ou a barreira! Ou seja, mesmo com a energia menor que o máximo do potencial **a partícula ainda pode atravessar a barreira!**. Esse fenômeno da mecânica quântica é conhecido como **tunelamento**. No caso em que a energia é maior que o máximo do potencial, a partícula também possui **probabilidade não nula de ser refletida!**.

---

# **Prática**

---

## **Requisitos do Código**
Primeiramente, temos três bibliotecas para construção desse código, sendo:

- numpy
- matplotlib.pyplot
- ipywidgets
- IPython.display
- scipy.sparse
- time

Elas foram utilizadas para simplificação de cálculos matemáticos, construções gráficas, criação de interfaces interativas e animações, respectivamente.

---

## **Explicação do código**

Para visualizar as propriedades calculadas anteriormente, foram construídos três gráficos:

- Azul para a função de onda,
- Vermelho para a barreira função delta de Dirac,
- Roxo para probabilidade de reflexão,
- Verde para probabilidade de transmissão.

Dessa forma, através dos widgets "Força ($α$)", "Momento ($k_0$)" e Tempo, podemos visualizar a animação da interação da função de onda e barreira de potencial.

---

## **Código**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.sparse import diags
from scipy.sparse.linalg import factorized
import time

# Cache e Parâmetros Físicos
cache = {
    'alpha_atual': None,
    'k0_atual': None,
    'historico_psi': None,
    'x': None,
    'L': 50.0,
    'N': 1500,
    'dt': 0.05,
    'dx': None
}

def simular_espalhamento(alpha, k0, frames=300, passos_por_frame=3):
    L, N, dt = cache['L'], cache['N'], cache['dt']
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    cache['x'], cache['dx'] = x, dx

    # Potencial da Barreira (Delta de Dirac)
    V_real = np.zeros(N)
    V_real[np.argmin(np.abs(x))] = alpha / dx

    # Potencial Absorvedor Complexo (CAP)
    # Absorve tudo que passar de x = 30 ou x = -30
    V_imaginario = np.zeros(N, dtype=complex)
    x_cap = 30.0
    for i in range(N):
        if x[i] < -x_cap:
            V_imaginario[i] = -1j * 0.2 * (abs(x[i]) - x_cap)**3
        elif x[i] > x_cap:
            V_imaginario[i] = -1j * 0.2 * (x[i] - x_cap)**3

    V_total = V_real + V_imaginario

    # Hamiltoniano com a borda absorvedora
    diag_princ = np.ones(N) * (1.0 / dx**2) + V_total
    diag_adj = np.ones(N-1) * (-0.5 / dx**2)
    H = diags([diag_adj, diag_princ, diag_adj], [-1, 0, 1])

    # Matrizes de Crank-Nicolson
    I = diags([np.ones(N)], [0])
    A = (I + 0.5j * dt * H).tocsc()
    B = (I - 0.5j * dt * H).tocsc()
    resolver_A = factorized(A)

    # Condição Inicial
    sigma = np.sqrt(2.0)
    psi = np.exp(-0.25 * ((x - -15.0) / sigma)**2) * np.exp(1j * k0 * x)
    psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx)

    historico = [np.abs(psi)**2]

    for _ in range(frames):
        for _ in range(passos_por_frame):
            psi = resolver_A(B.dot(psi))
        historico.append(np.abs(psi)**2)

    return historico

# Configuração da Interface
plt.ioff()
fig, ax = plt.subplots(figsize=(10, 5))

linha_psi, = ax.plot([], [], color='blue', linewidth=2)
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.text(0.5, 0.45, r'Barreira $\delta(x)$', color='red', fontsize=12)

texto_refletido = ax.text(-25, 0.4, '', fontsize=12, color='purple', fontweight='bold')
texto_tunelado = ax.text(10, 0.4, '', fontsize=12, color='green', fontweight='bold')

# Limitamos a visualização para esconder a "esponja" (que atua após o 30)
ax.set_xlim(-30, 30)
ax.set_ylim(0, 0.5)
ax.set_xlabel('Posição (x)')
ax.set_ylabel('Densidade de Probabilidade')
ax.grid(True, alpha=0.3)

area_preenchida = None
saida_grafica = widgets.Output()

# Função de Atualização da Tela
def atualizar_grafico(*args):
    global area_preenchida
    alpha = slider_alpha.value
    k0 = slider_k0.value
    tempo = slider_tempo.value

    # Recalcula a física se os parâmetros mudarem
    if cache['alpha_atual'] != alpha or cache['k0_atual'] != k0:
        with saida_grafica:
            clear_output(wait=True)
            print("Calculando nova simulação quântica... Aguarde um segundo.")
        cache['historico_psi'] = simular_espalhamento(alpha, k0)
        cache['alpha_atual'] = alpha
        cache['k0_atual'] = k0

    x = cache['x']
    dx = cache['dx']
    densidade = cache['historico_psi'][tempo]

    linha_psi.set_data(x, densidade)

    if area_preenchida is not None:
        area_preenchida.remove()
    area_preenchida = ax.fill_between(x, densidade, 0, color='blue', alpha=0.3)

    # Calcula probabilidade apenas na área visível
    prob_ref = np.sum(densidade[(x > -30) & (x < 0)]) * dx
    prob_tun = np.sum(densidade[(x > 0) & (x < 30)]) * dx

    texto_refletido.set_text(f'Refletido: {prob_ref:.1%}')
    texto_tunelado.set_text(f'Tunelado: {prob_tun:.1%}')
    ax.set_title(f'Espalhamento Quântico | Força = {alpha:.1f} | Momento = {k0:.1f}')

    # Injeta a imagem no Output (Formato ideal para o Colab)
    with saida_grafica:
        clear_output(wait=True)
        display(fig)

# Widgets e Controles
slider_alpha = widgets.FloatSlider(min=0.0, max=10.0, step=0.5, value=3.0, description='Força (α):')
slider_k0 = widgets.FloatSlider(min=1.0, max=6.0, step=0.5, value=3.0, description='Momento (k0):')
slider_tempo = widgets.IntSlider(min=0, max=90, step=1, value=0, description='Tempo:', layout=widgets.Layout(width='400px'))

btn_play = widgets.Button(description='▶️ Reproduzir', button_style='success', layout=widgets.Layout(width='120px'))

slider_alpha.observe(atualizar_grafico, names='value')
slider_k0.observe(atualizar_grafico, names='value')
slider_tempo.observe(atualizar_grafico, names='value')

def iniciar_play(b):
    btn_play.disabled = True
    btn_play.description = 'Rodando...'
    try:
        # Se estiver no final, recomeça
        if slider_tempo.value >= slider_tempo.max - 1:
            slider_tempo.value = 0

        # Roda o loop forçando a atualização da interface do Colab
        for t in range(slider_tempo.value, slider_tempo.max, 2):
            slider_tempo.value = t
            atualizar_grafico()
            time.sleep(0.01)
    finally:
        btn_play.disabled = False
        btn_play.description = '▶️ Reproduzir'

btn_play.on_click(iniciar_play)

controles_fisica = widgets.HBox([slider_alpha, slider_k0])
controles_tempo = widgets.HBox([btn_play, slider_tempo])
painel_completo = widgets.VBox([controles_fisica, controles_tempo, saida_grafica])

display(painel_completo)
atualizar_grafico()